<a href="https://colab.research.google.com/github/stevenolanecon/7002LBSAI/blob/main/notebooks/week10_data_exercises.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Before you start:**

- Go to **File → Save a copy in Drive**. Until you do this, your work only exists in this browser tab – you can't save changes back to GitHub, and closing the tab or losing the session will lose your work.
- Turn off Colab's autocomplete: **Tools → Settings → Editor → uncheck "Show context-powered code completions."** These exercises are meant to be worked through yourself – treat this the same as switching off a calculator's solver mode in an exam. It's a setting on your own Google account, not something built into this notebook, so you'll need to do it once per account.

# Week 10 – Data Exercises: Prediction

Adapted from Bekes & Kezdi, *Data Analysis for Business, Economics, and Policy* – drawing on both the model-building/cross-validation chapter (Ch13) and the LASSO/model-complexity chapter (Ch14). No output shown here to check yourself against: the point is to practise building models of increasing complexity, cross-validating them, and – in Questions 4 and 5 – running LASSO yourself, on data you haven't seen the answer for.

Easier and/or shorter exercises are marked **[\*]**; harder and/or longer exercises are marked **[\*\*]**.

Two of the textbook's own exercises for this chapter are left out: one asks you to collect brand-new used-car data from scratch (doesn't fit a pre-loaded-data exercise), and one is a pure combinatorics problem about how many models are possible as the predictor count grows (a genuinely different kind of task – ask if you'd like it as a separate short exercise).

## Setup

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import patsy
from sklearn.model_selection import KFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LassoCV

cars_path = "https://raw.githubusercontent.com/stevenolanecon/7002LBSAI/main/data/used_cars_chicago.csv"
vienna_path = "https://raw.githubusercontent.com/stevenolanecon/7002LBSAI/main/data/hotels_vienna.csv"
hackney_path = "https://raw.githubusercontent.com/stevenolanecon/7002LBSAI/main/data/airbnb_hackney.csv"
london_path = "https://raw.githubusercontent.com/stevenolanecon/7002LBSAI/main/data/airbnb_london.csv"
cps_path = "https://raw.githubusercontent.com/stevenolanecon/7002LBSAI/main/data/cps_earnings_2014.csv"

cars = pd.read_csv(cars_path)
vienna = pd.read_csv(vienna_path).dropna(subset=['rating'])
hackney = pd.read_csv(hackney_path)
hackney = hackney.loc[:, ~hackney.columns.duplicated()]
london = pd.read_csv(london_path)
cps = pd.read_csv(cps_path)
cps = cps[cps['earn_per_hour'] > 0]
cars.head()

> **Syntax hint – a publication-style comparison table.** Whenever a question below asks you to compare models, present them side by side – one column per model, one row per variable or metric, N at the bottom:
>
> ```python
> def stars(p):
>     return '***' if p < 0.01 else '**' if p < 0.05 else '*' if p < 0.1 else ''
>
> def compare_models(models, names, variables):
>     rows = []
>     for param_name, label in variables:
>         row = {'Variable': label}
>         for name, m in zip(names, models):
>             row[name] = f"{m.params[param_name]:.3f}{stars(m.pvalues[param_name])}" if param_name in m.params.index else ''
>         rows.append(row)
>     rows.append({'Variable': 'N', **{name: str(int(m.nobs)) for name, m in zip(names, models)}})
>     return pd.DataFrame(rows)
> ```
>
> For questions comparing fit statistics (BIC, RMSE) rather than coefficients, a plain `pd.DataFrame` with one row per model and one column per statistic works just as well – the point is a table, not prose.

## Question 1 [\*]

Use `cars` (the used-cars case study). The workshop used 5-fold cross-validation. Instead, use a **different number of folds** (try 10).

1. Find the best model by cross-validation (reuse the workshop's five model formulas).
2. Carry out a prediction for a specific car of your choosing, including a **prediction interval**.
3. Discuss what you find and compare with the workshop's own 5-fold findings.

## Question 2 [\*]

Use `cars` again. Instead of all 281 cars, keep only **LE-type cars that are at least 5 years old** (`LE == 1` and `age >= 5`).

1. Consider a few regression models analogous to the ones used in the workshop (start simple, add complexity).
2. Find the best model by cross-validation and carry out a prediction, including a prediction interval.
3. Discuss what you find and compare it with the workshop's full-sample findings. (This subsample is much smaller than the full 281 – keep that in mind when judging how much to trust your model.)

## Question 3 [\*\*]

Use `vienna` (the hotels-Vienna data) and specify **five regression models of increasing complexity** with `price` as the outcome, of your own design.

1. Find the best model by cross-validation.
2. Carry out a prediction for a specific hotel of your choosing, including a prediction interval.
3. Discuss your findings.

## Question 4 [\*\*]

Use `hackney` (the Airbnb case study). Repeat the workshop's model-building and selection exercise – **including the LASSO step** – using `log(price)` as the target instead of the level.

1. Build the same models (M1–M6) with `ln_price = np.log(hackney['price'])` as the outcome, then run LASSO on the M6 feature set.
2. To compare fairly with the workshop's level-price results, back-transform your predictions (`np.exp(...)`) before computing RMSE on the price scale.
3. Compare your results to the workshop's own (level-price) findings.

> **Syntax hint – LASSO.** You've seen this in the workshop; the pipeline is the same regardless of which target you use:
>
> ```python
> y, X = patsy.dmatrices('ln_price ~ ' + M6_formula, data=hackney, return_type='dataframe')
> y = y.values.ravel()
> X = X.drop(columns=['Intercept'])
> Xs = StandardScaler().fit_transform(X)
> lasso = LassoCV(cv=5, random_state=20260921, max_iter=20000).fit(Xs, y)
> (lasso.coef_ != 0).sum()   # how many variables survived
> ```

## Question 5 [\*\*]

Repeat the whole Airbnb exercise – model building **and LASSO** – on `london` (the full London dataset, not just Hackney), and add a set of binary variables for boroughs (`C(neighbourhood_cleansed)`).

1. Discuss your results and compare them to the Hackney-only case study.
2. Include a prediction-interval graph by, e.g., the number of people accommodated (`n_accommodates`), and compare it with the Hackney case study's version.
3. Note: `london` has ~30% missing values in `n_days_since` – any model that includes it will silently drop those rows. Keep an eye on how much your sample size shrinks.

## Question 6 [\*\*]

Use `cps` (the full 2014 CPS earnings data, `earn_per_hour` already constructed for you – 149,316 individuals, the same data behind Chapters 9–10). Build **several predictive models of increasing complexity** for earnings per hour, in both levels and logs (`np.log(cps['earn_per_hour'])`).

1. Compare model performance using **BIC** and **5-fold cross-validated RMSE** (remember: if you cross-validate a log model, back-transform predictions with `np.exp()` before computing RMSE, so it's comparable to the level models).
2. Take a 30% holdout set (`train_test_split(cps, test_size=0.3)`) and compare all your models' performance on it too.
3. Discuss the relationship between model complexity and performance – does more complexity keep helping, or does it plateau (or get worse) at some point?

*Your conclusion on how model complexity relates to performance here:*